# SAHI Batch Inference & Benchmarking — HuggingFace Models

This notebook demonstrates and benchmarks three inference strategies for HuggingFace object-detection models in SAHI:

| Strategy | How it works |
|---|---|
| **Sequential** | One image at a time (baseline) |
| **SAHI Native Batch** | `perform_batch_inference` — single processor + model forward pass |
| **HF Pipeline Batch** | `transformers.pipeline(batch_size=N)` with `KeyDataset` |

We also compare NMS postprocessing backends: **numpy** vs **torchvision**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obss/sahi/blob/feat-batch-inference/demo/batch_inference_benchmark.ipynb)

## 0. Setup

In [ ]:
# Install SAHI from the feat-batch-inference branch
!pip install -q git+https://github.com/obss/sahi.git@feat-batch-inference
# Pin transformers < 5.0 — RT-DETRv2 processor is broken in transformers 5.x
# (pixel values not normalized correctly, affects raw HF too, not just SAHI)
!pip install -q "transformers>=4.42.0,<5.0.0" timm datasets

# Verify
import transformers

from sahi.models.huggingface import HuggingfaceDetectionModel

assert hasattr(HuggingfaceDetectionModel, "_SIGMOID_CLS_PREFIXES"), (
    "Old SAHI version! Run: Runtime → Restart runtime, then re-run all cells."
)
print(f"transformers: {transformers.__version__}")
print("SAHI installed correctly")

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon MPS available")
else:
    device = "cpu"
    print("Running on CPU — benchmarks will still work, GPU values will be faster")

print(f"Using device: {device}")

In [ ]:
import os
import time

import numpy as np
from IPython.display import Image, display

from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
from sahi.utils.cv import read_image
from sahi.utils.file import download_from_url

In [ ]:
# Download sample images
os.makedirs("demo_data", exist_ok=True)

download_from_url(
    "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/small-vehicles1.jpeg",
    "demo_data/small-vehicles1.jpeg",
)
download_from_url(
    "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/terrain2.png",
    "demo_data/terrain2.png",
)

# We'll use small-vehicles1 for all inference demos
IMAGE_PATH = "demo_data/small-vehicles1.jpeg"
print("Images ready.")

## 1. Load the Model

We use **RT-DETR v2 (R18)** — the smallest RT-DETR v2 variant. Any HuggingFace `object-detection` model works.

> **Tip**: Set `HF_TOKEN` if you see authentication errors.

In [ ]:
MODEL_PATH = "PekingU/rtdetr_v2_r18vd"
# Alternatives: "PekingU/rtdetr_v2_r50vd", "facebook/detr-resnet-50"

# Uncomment if needed:
# os.environ["HF_TOKEN"] = "hf_..."

detection_model = AutoDetectionModel.from_pretrained(
    model_type="huggingface",
    model_path=MODEL_PATH,
    confidence_threshold=0.5,
    image_size=640,
    device=device,
)
print(f"Model loaded: {MODEL_PATH}")
print(f"Categories: {len(detection_model.category_mapping)}")

## 2. Sequential Inference (Baseline)

Process images one by one — the traditional approach.

> **Note**: Batch inference shines on **GPU** where the batch is parallelized across CUDA cores. On CPU, batch can actually be *slower* due to padding overhead and larger tensors. If you're on CPU, the speedup column may show < 1x — that's expected.

In [ ]:
result = get_prediction(IMAGE_PATH, detection_model)
result.export_visuals(export_dir="demo_data/", file_name="sequential_result")
display(Image("demo_data/sequential_result.png"))
print(f"Detections: {len(result.object_prediction_list)}")

In [ ]:
# Benchmark: sequential single-image inference
image_np = read_image(IMAGE_PATH)
BATCH_SIZES = [2, 4, 8, 16]  # test multiple batch sizes

# Warm-up (2 runs)
for _ in range(2):
    detection_model.perform_inference(image_np)
    detection_model.convert_original_predictions()

# Measure per-image time for sequential
REPEATS = 3
seq_times = []
for _ in range(REPEATS):
    t0 = time.perf_counter()
    detection_model.perform_inference(image_np)
    detection_model.convert_original_predictions()
    seq_times.append(time.perf_counter() - t0)
seq_per_img = min(seq_times)  # best of 3

print(f"Sequential: {seq_per_img * 1000:.1f} ms/image (best of {REPEATS})")

## 3. SAHI Native Batch Inference — Scaling Test

Test batch sizes 2, 4, 8, 16 to see how throughput scales on your hardware.

On **GPU (T4/V100)**: expect 2-5x speedup as batch size grows.  
On **CPU**: expect ~1x or slower (padding overhead dominates).

In [ ]:
# Quick demo: batch of 2 different images
img1 = read_image("demo_data/small-vehicles1.jpeg")
img2 = read_image("demo_data/terrain2.png")

detection_model.perform_batch_inference([img1, img2])
detection_model.convert_original_predictions(
    shift_amount=[[0, 0], [0, 0]],
    full_shape=[[img1.shape[0], img1.shape[1]], [img2.shape[0], img2.shape[1]]],
)

per_image_preds = detection_model.object_prediction_list_per_image
for i, preds in enumerate(per_image_preds):
    print(f"  Image {i}: {len(preds)} detections")

In [ ]:
# Benchmark across batch sizes
print(f"{'Batch Size':>12} {'Total (s)':>10} {'ms/image':>10} {'Speedup':>10}")
print("-" * 46)

# Sequential baseline row
seq_total_8 = seq_per_img * 8
print(f"{'seq (1x8)':>12} {seq_total_8:>10.3f} {seq_per_img * 1000:>10.1f} {'1.00x':>10}")

batch_results = {}
for bs in BATCH_SIZES:
    images = [image_np] * bs

    # Warm-up
    detection_model.perform_batch_inference(images[:2])

    best = float("inf")
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        detection_model.perform_batch_inference(images)
        detection_model.convert_original_predictions(
            shift_amount=[[0, 0]] * bs,
            full_shape=[list(image_np.shape[:2])] * bs,
        )
        elapsed = time.perf_counter() - t0
        best = min(best, elapsed)

    per_img = best / bs
    speedup = seq_per_img / per_img
    batch_results[bs] = {"total": best, "per_img": per_img, "speedup": speedup}
    print(f"{'batch ' + str(bs):>12} {best:>10.3f} {per_img * 1000:>10.1f} {speedup:>9.2f}x")

# Pick best batch size for later comparison
best_bs = max(batch_results, key=lambda k: batch_results[k]["speedup"])
elapsed_batch = batch_results[best_bs]["total"]
print(f"\nBest batch size: {best_bs} ({batch_results[best_bs]['speedup']:.2f}x speedup)")

## 4. HuggingFace `pipeline` Comparison

The high-level `transformers.pipeline("object-detection")` API as a comparison point.

In [ ]:
from PIL import Image as PILImage
from transformers import pipeline

PIPELINE_BS = BATCH_SIZES[-1]  # use largest batch size
detector = pipeline(
    "object-detection",
    model=MODEL_PATH,
    device=0 if device == "cuda" else -1,
)
print(f"Pipeline loaded (batch_size={PIPELINE_BS} for benchmark)")

In [ ]:
# Run pipeline on a single image
pil_img = PILImage.open(IMAGE_PATH)
result = detector(pil_img, threshold=0.5)
print(f"Pipeline detections: {len(result)}")
for det in result[:5]:
    print(f"  {det['label']:<20} score={det['score']:.3f}  box={det['box']}")

In [ ]:
# Benchmark: sequential pipeline calls (apples-to-apples comparison)
# Warm-up
detector(pil_img, threshold=0.5)

best_pipeline = float("inf")
for _ in range(REPEATS):
    t0 = time.perf_counter()
    for _ in range(PIPELINE_BS):
        detector(pil_img, threshold=0.5)
    best_pipeline = min(best_pipeline, time.perf_counter() - t0)

elapsed_pipeline = best_pipeline
per_img_pipeline = elapsed_pipeline / PIPELINE_BS * 1000
speedup_pipeline = seq_per_img / (elapsed_pipeline / PIPELINE_BS)

print(
    f"HF pipeline ({PIPELINE_BS} images): {elapsed_pipeline:.3f}s  "
    f"({per_img_pipeline:.1f} ms/image, {speedup_pipeline:.2f}x vs SAHI sequential)"
)

## 5. Benchmark Summary

In [ ]:
print(f"Device: {device}")
print(f"Model:  {MODEL_PATH}\n")

print(f"{'Strategy':<30} {'ms/image':>10} {'Speedup':>10}")
print("-" * 52)
print(f"{'Sequential (1-by-1)':<30} {seq_per_img * 1000:>10.1f} {'1.00x':>10}")

for bs in BATCH_SIZES:
    r = batch_results[bs]
    print(f"{'SAHI batch (bs=' + str(bs) + ')':<30} {r['per_img'] * 1000:>10.1f} {r['speedup']:>9.2f}x")

print(
    f"{'HF pipeline (bs=' + str(BATCH_SIZES[-1]) + ')':<30} {elapsed_pipeline / BATCH_SIZES[-1] * 1000:>10.1f} {seq_per_img * BATCH_SIZES[-1] / elapsed_pipeline:>9.2f}x"
)

if device == "cpu":
    print("\n** Running on CPU — batch speedup is expected on GPU (T4/V100/A100). **")
    print("** To test on GPU: Runtime → Change runtime type → T4 GPU **")

## 6. Postprocessing Backend Comparison

SAHI supports three postprocessing backends for NMS/NMM:

| Backend | When to use |
|---|---|
| `numpy` | Default — no extra deps, works everywhere |
| `numba` | CPU-heavy workloads — JIT-compiled, fast after warm-up |
| `torchvision` | GPU available — uses CUDA NMS kernel |

In [ ]:
from sahi.postprocess.backends import get_postprocess_backend, set_postprocess_backend
from sahi.postprocess.combine import nms

# Generate synthetic predictions: 1000 random boxes
rng = np.random.default_rng(42)
N = 1000
x1 = rng.uniform(0, 900, N).astype(np.float32)
y1 = rng.uniform(0, 900, N).astype(np.float32)
x2 = (x1 + rng.uniform(10, 100, N)).astype(np.float32)
y2 = (y1 + rng.uniform(10, 100, N)).astype(np.float32)
scores = rng.uniform(0.1, 1.0, N).astype(np.float32)
cats = rng.integers(0, 5, N).astype(np.float32)
preds = np.stack([x1, y1, x2, y2, scores, cats], axis=1)

backends_to_test = ["numpy"]
try:
    import torchvision  # noqa: F401

    backends_to_test.append("torchvision")
except ImportError:
    pass
try:
    import numba  # noqa: F401

    backends_to_test.append("numba")
except ImportError:
    pass

print(f"Backends available: {backends_to_test}")
print(f"Running NMS on {N} predictions\n")

REPS = 20
results_backend = {}
for backend in backends_to_test:
    set_postprocess_backend(backend)
    # Warm-up
    nms(preds, match_threshold=0.5)
    t0 = time.perf_counter()
    for _ in range(REPS):
        keep = nms(preds, match_threshold=0.5)
    elapsed = (time.perf_counter() - t0) / REPS * 1000
    results_backend[backend] = (elapsed, len(keep))
    print(f"  {backend:<12} {elapsed:6.2f} ms/call   kept {len(keep)} boxes")

# Reset to auto
set_postprocess_backend("auto")
print(f"\nCurrent backend (auto): {get_postprocess_backend()}")

## 7. Batch Inference + Sliced Prediction

The batch-enabled model plugs directly into `get_sliced_prediction`.  
Internally, SAHI uses `perform_batch_inference` to process multiple slices at once.

In [ ]:
result = get_sliced_prediction(
    IMAGE_PATH,
    detection_model,
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=True,
    postprocess_type="GREEDYNMM",
    postprocess_match_threshold=0.5,
)

result.export_visuals(export_dir="demo_data/", file_name="sliced_batch_result")
display(Image("demo_data/sliced_batch_result.png"))
print(f"Detections after sliced inference + postprocess: {len(result.object_prediction_list)}")

## Summary

- **GPU**: Batch inference gives real speedups (2-5x on T4, more on V100/A100) by parallelizing the forward pass across CUDA cores.
- **CPU**: Batch is typically slower due to padding overhead — sequential is optimal on CPU.
- **SAHI native batch** (`perform_batch_inference`) is the recommended approach: single GPU forward pass, integrates with the full SAHI pipeline.
- **HF pipeline with KeyDataset** is a good standalone alternative; use `Dataset + KeyDataset` to avoid the batch-size bug.
- **Postprocessing backends**: `torchvision` gives best GPU NMS performance; `numpy` is the safe default.

### Next steps

- [Postprocessing backends guide](https://obss.github.io/sahi/postprocess/backends/)
- [SAHI HuggingFace inference demo](https://colab.research.google.com/github/obss/sahi/blob/main/demo/inference_for_huggingface.ipynb)
- [SAHI GitHub](https://github.com/obss/sahi)